# 05.6 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/06-hyperparameter-tuning/exercises.ipynb)

Exercise 1 is on paper. The others need the churn table; run the setup cell first.

## Exercise 1: count the fits (paper)

A grid with `n_estimators` ∈ {100, 200, 400}, `learning_rate` ∈ {0.01, 0.05}, `num_leaves` ∈ {4, 8, 16, 32}, with 5-fold CV.

a) How many settings? b) How many model fits, including the final refit? c) If each fit takes 0.3 s on one core, how long on one core? On 8 cores?

*Your answer here.*

In [ ]:
import os, logging
os.environ["PYTHONWARNINGS"] = "ignore"
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train)
auc_logreg = roc_auc_score(y_test, logreg.predict_proba(X_test)[:, 1])
print(f"Logistic regression baseline, test AUC: {auc_logreg:.3f}")

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # the same 5 folds for every setting

## Exercise 2: a third dimension

Add `min_child_samples: [10, 40, 80]` to the grid from section 3. How many fits now? Does the best CV AUC change by more than the fold std (≈ 0.03)? Time it with `%%time`.

In [ ]:
# your code here

## Exercise 3: tune a random forest

Use `RandomizedSearchCV` with `n_iter=15` on `RandomForestClassifier(n_estimators=200, random_state=42)` over `max_depth` (3–15), `min_samples_leaf` (1–50) and `max_features` (`["sqrt", 0.5, 1.0]`). Compare its best CV AUC with tuned LightGBM.

In [ ]:
# your code here

## Exercise 4: is Optuna's answer stable?

Run the Optuna study three times with seeds 0, 1, 2 and 15 trials each. Print the best settings and best CV AUC. Are the settings similar? Are the scores?

In [ ]:
# your code here